<a href="https://colab.research.google.com/github/labphonlab/phonetics-research-statistics-support/blob/main/notebooks/10_pipeline_targets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このバッジから開くと、設定なしでそのまま実行できます。
書き換えを残したいときは **ファイル → ドライブにコピーを保存** を一度行ってください。


# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 付録：Rを単体で回すより、パイプラインに組んだほうが実務的か


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本ファイルの公開用URLを、みだりに他の人へ共有しないこと。

本ファイルの公開用URLにアクセスすると、お手元のGoogleアカウントに本ファイルのコピーが作成される。
そのコピーはあなただけのファイルとして生成されるため、自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照: Landau, W. M. (2021). The targets R package: a dynamic Make-like function-oriented pipeline toolkit for reproducibility and high-performance computing. *Journal of Open Source Software*, 6(57), 2959。

■ 問い：ノートブックを毎回上から実行するのと、パイプラインを組むのはどちらが良いか：

本書の第1〜9回は、学習用に「1つのノートブックを上から実行する」形をとっている。初学者にとってはこれが最短距離で、章をまたぐ依存を気にせず読めるという利点がある。

しかし、実際の研究で同じ前処理（第1回）→複数の統計モデル（第3〜9回）→図表出力、という工程を**繰り返す**ようになったら話は別である。毎回全セルを再実行すると、

1. 変更していない前処理まで毎回再計算して時間を無駄にする
2. どのモデルがどのデータ・どのバージョンのコードから作られたか追跡できなくなる（再現性の喪失）
3. 複数の解析を並行して試すとき、ノートブックのセル順序に解析の依存関係を人間が手で管理することになる

という問題が出る。**答えは「単体実行→最後にパイプライン化」の順**である。本書のように1トピック1ノートブックで学ぶ段階では単体実行のままでよいが、本格的な研究に入ったら`targets`パッケージへの移行を検討するとよい。

In [ ]:
# ⏳ 数十秒〜数分。
install.packages("targets", repos = "https://cloud.r-project.org")
library(targets)


■ targetsパッケージの考え方：

`targets`は「入力が変わっていない処理は再実行しない」ことを保証するRのパイプラインツールである（GNU Makeの考え方をRの関数に適用したもの）。プロジェクト直下に`_targets.R`という設定ファイルを1つ置き、処理をターゲット（関数呼び出し）の連鎖として宣言する。

In [ ]:
# 以下はプロジェクトのルートに保存する _targets.R の中身のイメージ（このセル自体はテキスト例示用）
pipeline_example <- '
library(targets)
library(tarchetypes)

tar_option_set(packages = c("tidyverse", "lme4", "brms"))

list(
  # 第1回: データ読み込み・前処理（このターゲットの出力はvot_data.csvの内容が変わらない限り再計算されない）
  tar_target(vot_data_file, "vot_f0_synthetic.csv", format = "file"),
  tar_target(vot_data, read_csv(vot_data_file, show_col_types = FALSE) %>%
    mutate(
      speaker = factor(speaker), word = factor(word),
      gender = factor(gender),
      condition = factor(condition, levels = c("voiced", "voiceless"))
    )
  ),

  # 第5回: 線形混合効果モデル（vot_dataが変わったときだけ再フィットされる）
  tar_target(model_lmm, lmer(
    VOT ~ condition + (1 + condition | speaker) + (1 | word),
    data = vot_data
  )),

  # 第8回: ベイズモデル（重い計算なので、変更がなければ絶対に再実行したくない）
  tar_target(model_brm, brm(
    VOT ~ condition + (1 + condition | speaker) + (1 | word),
    data = vot_data, chains = 4, iter = 2000, warmup = 1000, seed = 42
  )),

  # 図表出力（モデルが変わったときだけ再描画される）
  tar_target(fig_vot_violin, {
    p <- ggplot(vot_data, aes(x = condition, y = VOT, fill = condition)) +
      geom_violin(alpha = 0.5) + theme_minimal()
    ggsave("fig_vot_violin.png", p, width = 6, height = 4)
    "fig_vot_violin.png"
  }, format = "file")
)
'
cat(pipeline_example)

# 上のテキストをそのまま_targets.Rとして書き出す。演習でtar_make()を動かす
# ときは、これを実行してから進めればよい（手でファイルを作る必要はない）。
cat(pipeline_example, file = "_targets.R")


■ 何が嬉しいのか：

- `tar_make()`を実行すると、変更されたターゲットとその下流だけが再計算される。第8回のbrmsモデル（数分かかる）は、データを変えない限り2回目以降は一瞬でキャッシュから返る。
- `tar_visnetwork()`で依存関係をグラフとして可視化でき、「このモデルはどの前処理から来ているか」が一目でわかる。
- CI（継続的インテグレーション）に載せやすい。ノートブックのセルを手で順番に実行する必要がなくなる。

**ただし、これは学習段階で必要なものではない。** 本書の第1〜9回のように「1トピックを1回動かして理解する」場面では、単体のColabノートブックのほうがシンプルで学習効率が良い。パイプライン化は、同じ解析を何度も回す実際の研究フェーズに入ってから検討すればよい。

■ 演習：

1. 本書の第1回・第5回・第8回のコードを`_targets.R`に移し、`tar_make()`を実行してパイプラインが動くことを確認せよ。
2. `vot_f0_synthetic.csv`を書き換えて`tar_make()`を再実行し、どのターゲットが再計算されるかを`tar_visnetwork()`で確認せよ。